### 01 — Model Training

This notebook reproduces the Gradient Boosting correction model described in
the original heuristic-boosted EF estimation pipeline. Since the original
model artifact is not portable from its training environment (wandb), it is
retrained here from precomputed inference features using the same
hyperparameters and feature set reported in the original work.

The reported test MAE (heuristic: 6.42%, corrected: 4.69%) is used as a
validation gate: if this notebook cannot reproduce those figures within a
reasonable margin, no downstream analysis (clinical classification, SHAP
explainability) should proceed.

**Outputs of this notebook:**
- A trained Gradient Boosting correction model (`models/gb_correction.joblib`)
- A processed dataset with corrected EF, signed error, and outlier flags
  (`data/processed/inference_with_corrections.csv`), used by all subsequent
  notebooks.

#### Load data and compute signed error

In [75]:
import pandas as pd
import numpy as np

data = pd.read_csv("../data/raw/inference_feat.csv")
data['error'] = data['true_ef'] - data['predicted_ef']

df = data[ (data['split'] == 'TRAIN') | (data['split'] == 'VAL')]
test = data[data['split'] == 'TEST']

#### Outlier filtering (Z-score)

Samples with an absolute Z-score greater than 2 on the signed error are
considered outliers, following the original methodology. To avoid data
leakage, the mean and standard deviation used for the Z-score are computed
using the TRAIN split only, then applied to filter TRAIN and VAL together.

In [76]:
mu = df.loc[df['split'] == 'TRAIN', 'error'].mean()
sigma = df.loc[df['split'] == 'TRAIN', 'error'].std()

z = (df['error'] - mu) / sigma
filtered_df = df[z.abs() < 2]

In [77]:
df = filtered_df

#### Prepare features and target

In [78]:
df.drop(columns=['true_ef', 'predicted_ef', 'filename', 'split'], inplace=True)

X, y = df.drop('error', axis=1), df['error']

#### Feature set

The three predictors identified as most informative in the original work:
ventricular length ratio, volume ratio, and the standard deviation of the
Dice similarity coefficient (temporal segmentation consistency).

In [79]:
features = ['volume_ratio', 'length_ratio', 'dice_overlap_std']

#### Train the correction model

`GradientBoostingRegressor` with the hyperparameters reported in the original
work (learning rate 0.01, max depth 4, 200 estimators). The model is fit on
the filtered TRAIN+VAL set and evaluated on the untouched TEST set.

In [80]:
from sklearn.ensemble import GradientBoostingRegressor

reg = GradientBoostingRegressor(learning_rate=0.01, max_depth=4, n_estimators=200, random_state=1)
reg.fit(X[features], y)

X_test = test[features]
y_test = test['error']

y_pred = reg.predict(X_test)

In [81]:
test = data[data['split'] == 'TEST'].copy()
test.loc[:, 'corrected_ef'] = test['predicted_ef'] + y_pred

#### Validation gate: reproduce reported MAE

In [82]:
from sklearn.metrics import mean_absolute_error

mae_heuristic = mean_absolute_error(test['true_ef'], test['predicted_ef'])
mae_corrected = mean_absolute_error(test['true_ef'], test['corrected_ef'])

print(f"MAE heurístico: {mae_heuristic:.2f}%  (paper: 6.42%)")
print(f"MAE corregido:  {mae_corrected:.2f}%  (paper: 4.69%)")

MAE heurístico: 6.43%  (paper: 6.42%)
MAE corregido:  4.70%  (paper: 4.69%)


In [83]:
data['error_pred'] = reg.predict(data[features])
data['corrected_ef'] = data['predicted_ef'] + data['error_pred']

In [84]:
mask_outlier = np.abs((data['error'] - mu) / sigma) >= 2
data['is_outlier'] = mask_outlier & (data['split'] != 'TEST')

#### Save trained model and enriched dataset

In [85]:
import joblib
joblib.dump(reg, "../models/gb_correction.joblib")

data.to_csv("../data/processed/inference_with_corrections.csv", index=False)